# Cell-Type Annotation

**REQUIRED DAY 2**

## Load your checkpoint

Fresh kernel -- loading back the `adata` saved at the end of [07_dimensionality_reduction_and_clustering.ipynb](07_dimensionality_reduction_and_clustering.ipynb) (PCA, neighbors, UMAP, `leiden` clusters already computed).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_07_clustered.h5ad")
adata


## From clusters to cell types

Clustering (previous notebook) gives you groups of transcriptionally similar cells. It does not tell you what those groups *are* — that's a separate, biological step: matching each cluster's characteristic genes against known marker genes for expected cell types.

Today's sample is PBMCs (peripheral blood mononuclear cells), so the expected cast is roughly: T cells, B cells, NK cells, and monocytes. Here's a reference panel — **don't jump to it yet**, the next section asks you to work out cluster identity yourself first, from the actual genes your clusters produce:

| Marker gene | Cell type |
| --- | --- |
| CD3D, CD3E | T cells (all) |
| CD8A | Cytotoxic T cells |
| MS4A1, CD19 | B cells |
| NKG7, GNLY | NK cells |
| LYZ, CD14 | Classical monocytes |
| FCGR3A | Non-classical monocytes |
| PPBP | Platelets |

## Annotation backed by a statistic, not a glance

It's tempting to color the UMAP by one marker gene, squint, and declare a cluster's identity. Do this instead — find each cluster's actual top differentially expressed genes and check them against the marker panel:

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

## Work it out yourself before checking the panel

For at least two of your clusters: look at its real top genes in the plot above, pick one or two that *aren't* already in the reference panel, and actually search for them (GeneCards, PanglaoDB, CellMarker DB, or a general web search all work) — what cell type or state are they known markers of? Write your hypothesis for those clusters below, before you look at the dotplot in the next cell. This is closer to what real annotation work looks like than matching against a list someone handed you — the list above happens to cover this sample's most common types, but it won't cover every dataset you'll ever annotate.

*(Fill in: for each cluster you researched, the gene(s) you searched and what you found.)*

## Now check yourself against the reference panel

In [ ]:
PBMC_MARKERS = ["CD3D", "CD3E", "CD8A", "MS4A1", "CD19", "NKG7", "GNLY", "LYZ", "CD14", "FCGR3A", "PPBP"]
sc.pl.dotplot(adata, PBMC_MARKERS, groupby="leiden")

`rank_genes_groups` runs a statistical test (Wilcoxon rank-sum, by default) for every gene, per cluster, against the rest of the data — you get an actual ranked, tested list, not an impression. That's a real, checkable answer to "is each cell-type annotation backed by a statistical marker test, not just a colored UMAP that looks right?"

Once you're satisfied, label the clusters:

In [ ]:
adata.obs["cell_type"] = adata.obs["leiden"].map({
    "0": "CD4 T cells",
    "1": "CD14+ Monocytes",
    # ... fill in based on what rank_genes_groups actually shows you
})
sc.pl.umap(adata, color="cell_type")

## Why the full genome index mattered (the payoff)

Back in [03_raw_data_fastq_to_counts.md](03_raw_data_fastq_to_counts.md), the plan was to pre-build a **full** genome index rather than restrict it to one chromosome, because the marker panel above spans chromosomes 1, 2, 4, 5, 11, 12, 16, and 19 (GRCh37 coordinates). If the index had been restricted to save build time, some of these markers would have been silently unmappable, and you would have seen fewer expected cell types today for a reason that had nothing to do with the actual biology of this sample.

## Further reading

- [Single-cell best practices — Annotation](https://www.sc-best-practices.org/cellular_structure/annotation.html)
- [scanpy: `rank_genes_groups` documentation](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html)